In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import time
import os
import json
import ast
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

import sys
import tqdm
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [2]:
def obs_to_fragment_man(nodes, configs):
    fragment_map = np.zeros((30,30))
    for ii, node in enumerate(nodes):
        fragment_map[node[0]-2+3:node[0]+3+3,node[1]-2+3:node[1]+3+3] += 1*configs[ii]
    return np.minimum(1,fragment_map[3:27,3:27])
    
def extract_episodes():
    files = os.listdir(os.getcwd()+ "/data/raw_data")
    all_data = []
    targets = []
    sample_id = 0
    for episode in tqdm.tqdm(range(len(files))):
        with open(f"data/raw_data/{files[episode]}") as f:
            data = json.load(f)
        params = data["steps"][0][0]["info"]["replay"]["params"]
        sap_range = params["unit_sap_range"]
        move_cost = params["unit_move_cost"]
        match = []
        for step in range(505):
            step_pos = []
            step_eng = []
            step_acs = []
            energy_map = np.array(data["steps"][step][0]["info"]["replay"]["observations"][0]["map_features"]["energy"])
            tile_map = np.array(data["steps"][step][0]["info"]["replay"]["observations"][0]["map_features"]["tile_type"])
            energy_map[tile_map==1] -= params["nebula_tile_energy_reduction"]
            energy_map = (energy_map-energy_map.mean())/(energy_map.std()+1e-8)
            tile_map[tile_map==1] = 0
            tile_map[tile_map==2] = 1
            
            nodes = data["steps"][step][0]["info"]["replay"]["observations"][0]["relic_nodes"]
            configs = data["steps"][step][0]["info"]["replay"]["observations"][0]["relic_node_configs"]
            fragment_map = obs_to_fragment_man(nodes, configs)
            player = 0
            obs = json.loads(data["steps"][step][player]["observation"]["obs"])
            positions = obs["units"]["position"]
            #print(obs["units"])
            masks = [obs["units_mask"]]
            own_ids = np.arange(16)[masks[0][0]]
            own_positions = np.array(positions[0])[masks[0][0]]
            own_positions_map = np.zeros((24,24))
            own_positions_map[own_positions[:,0],own_positions[:,1]] = 1
            enemy_positions = np.array(positions[1])[masks[0][1]]
            enemy_positions_map = np.zeros((24,24))
            enemy_positions_map[enemy_positions[:,0],enemy_positions[:,1]] = 1
            actions = np.array(data["steps"][step+1][player]["action"])
            if enemy_positions.size!=0:
                for ii, pos in enumerate(own_positions):
                    unit = own_ids[ii]
                    eng = obs["units"]["energy"][player][unit]
                    if eng>=move_cost and np.max(np.abs((enemy_positions-pos)))<=8: # valid datapoint if in max sap_range
                        step_pos.append(pos)
                        step_eng.append(eng)
                        acs = actions[unit]
                        
                        step_acs.append(actions[unit])
            in_own_positions = own_positions_map.copy()
            in_enemy_positions = enemy_positions_map.copy()
            in_fragment_map = fragment_map.copy()
            X = np.concatenate((np.expand_dims(in_own_positions,axis=0), np.expand_dims(in_enemy_positions,axis=0), 
                                np.expand_dims(in_fragment_map,axis=0), np.expand_dims(tile_map,axis=0), np.expand_dims(energy_map,axis=0)),axis=0)
            sample = (torch.tensor(X), step_pos, step_acs)
            if step_pos:
                match.append(sample)
        torch.save((params, match), f"data/extracted_data/epi_{episode}")
#extract_episodes()

In [3]:
class PlaysDataset(torch.utils.data.Dataset):
    def __init__(self, data, num=1e4):
        self.num = int(num)
        self.data = data
        for i in range(int(num)):
            self.data.append() # take all files in the root directory
    def __len__(self):
        return self.num
    def __getitem__(self, idx):
        X_m, X_p, label = self.data[idx] # load the features of this sample
        return sX_m, X_p, label

In [4]:
def episode_to_samples(episode):
    params, match = episode
    r = params["unit_sap_range"]
    input_params = torch.tensor([(r-3)/5, params["unit_move_cost"]/4, (params["unit_sap_cost"]-30)/21, params["unit_sap_dropoff_factor"], params["unit_energy_void_factor"]])
    data = []
    for step in range(len(match)):
        sample = match[step]
        X, pos, eng, acs = sample
        for ii, p in enumerate(pos):
            X_sample = X.clone()
            sap_range_map = torch.zeros((1,24,24))
            sap_range_map[0,p[0]-r:p[0]+r+1,p[1]-r:p[1]+r+1] = 1
            in_pos_map = torch.zeros((1,24,24))
            in_pos_map[0,p[0],p[1]] = 1
            X_sample[0,p[0],p[1]] = 0
            X_sample = torch.cat((in_pos_map, sap_range_map, X_sample))
            X_params = torch.cat((torch.tensor(eng[ii]/400).unsqueeze(0), input_params))
            sample = (X_sample, X_params, acs[ii])
            data.append(sample)
    return data
    
def load_data(trainsize=0.9, testsize=0.1, seed=42, root="data/extracted_data"):
    files = os.listdir(root)
    file_indices = np.arange(len(files))
    np.random.shuffle(file_indices)
    data = []
    for i in tqdm.tqdm(range(int(len(files)/5)), file=sys.stdout, colour="green"): 
        data = data + episode_to_samples(torch.load(os.path.join(root, files[file_indices[i]]), weights_only=False))
    indices = np.arange(len(data))
    np.random.shuffle(indices)
    train_indices = indices[:int(trainsize*len(indices))]
    test_indices = indices[int(trainsize*len(indices)):]
    traindata, testdata = [], []
    print("Loading training data...")
    for i in tqdm.tqdm(range(len(train_indices)), colour="green"):
        traindata.append(data[train_indices[i]])
    print("Loading test data...")
    for i in tqdm.tqdm(range(len(test_indices))):
        testdata.append(data[test_indices[i]])
    return traindata, testdata


                    

In [5]:
trainsize = 0.9
testsize = 0.1
traindata, testdata = load_data(trainsize, testsize)

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:53<00:00,  3.71it/s]
Loading training data...


100%|█████████████████████████████████████████████████████████████████████| 208320/208320 [00:00<00:00, 2147647.28it/s]


Loading test data...


100%|███████████████████████████████████████████████████████████████████████| 23147/23147 [00:00<00:00, 1780406.28it/s]


In [6]:
def weights_init(m):
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)

class ActionOracle(torch.nn.Module):
    def __init__(self,n_maps, n_params):
        super().__init__()
        self.n_maps = n_maps
        self.n_params = n_params
        self.cnn = nn.Sequential(
                nn.Conv2d(self.n_maps, 16, kernel_size=1, padding=1),
                nn.ReLU(),
                #nn.MaxPool2d(2),
                nn.Conv2d(16, 16, kernel_size=5, padding=1),
                nn.ReLU(),
                nn.Conv2d(16, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(16, 8, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(8, 4, kernel_size=3, padding=1),
                nn.ReLU(),
                #nn.AvgPool2d(2),
                nn.Flatten(),
                nn.Linear(24*24*4, 128),
                nn.ReLU(),
            )
        self.ff = nn.Sequential(
                nn.Linear(128+self.n_params,64),
                nn.ReLU(),
                nn.Linear(64, 5),
                nn.ReLU(),
                nn.Softmax(),
        )
        self.cnn.apply(weights_init)
        self.ff.apply(weights_init)
        
    def forward(self, maps, params):
        cnn_out = self.cnn(maps)
        return self.ff(torch.cat((cnn_out, params),dim=-1))

In [8]:
def get_counts(trainloader):
    counts = torch.zeros((5))
    for X1, X2, y in trainloader:
        count = torch.unique(y[:,0], return_counts=True)[1]
        counts +=count
    return counts
#def test_calibration(model, testloader):
    
def test(model, testloader):
    accuracy_per_ac = {
        "0": [],
        "1": [],
        "2": [],
        "3": [],
        "4": [],
    }
    accuracies = []
    t2accuracies = []
    t3accuracies = []
    n_ac = []
    for X_maps, X_params, y in testloader:
        with torch.no_grad():
            X_maps, X_params, y = X_maps.to(torch.float32), X_params.to(torch.float32), y[:,0]
            out = model(X_maps, X_params)
            #print(out.shape)
            pred = torch.argmax(out.detach(), dim=1)
            #print(out[0])
            out[torch.arange(out.shape[0]),pred] = 0
            #print(out[0])
            pred2 = torch.argmax(out.detach(), dim=1)
            out[torch.arange(out.shape[0]),pred2] = 0
            pred3 = torch.argmax(out.detach(), dim=1)
            n_ac.append(torch.max(torch.unique(pred, return_counts=True)[1]).to(torch.float32))
            corrects = (1*(pred==y)).to(torch.float32)
            corrects2 = torch.clamp((1*(pred==y) + 1*(pred2==y)),0,1).to(torch.float32)
            corrects3 = torch.clamp((1*(pred==y) + 1*(pred2==y)+ 1*(pred3==y)),0,1).to(torch.float32)
            accuracies.append(corrects.mean())
            t2accuracies.append(corrects2.mean())
            t3accuracies.append(corrects3.mean())
            for ii, t in enumerate(y):
                accuracy_per_ac[str(t.item())].append(corrects[ii].item())
    return accuracy_per_ac, n_ac, accuracies, t2accuracies, t3accuracies
    
def train(traindata, testdata, run_name="test", batch_size=64, lr=0.001, n_epoch=100):
    batch_size = batch_size
    trainloader = torch.utils.data.DataLoader(traindata, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(testdata, batch_size=batch_size, shuffle=False)
    counts = get_counts(trainloader)
    weights = counts.sum()/counts
    #weights = torch.ones_like(counts)
    train_len = len(trainloader)
    lr = lr
    n_maps = 7
    n_params = 6
    t = 0
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"aimbot_{timestamp}"
    writer = SummaryWriter(f"runs/{run_name}")
    model = ActionOracle(n_maps, n_params)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epoch)
    for epoch in tqdm.tqdm(range(n_epoch)):
        losses = []
        accuracies= []
        n_ac = []
        for batch_num, (X_maps, X_params, y) in enumerate(trainloader):
            X_maps, X_params, y = X_maps.to(torch.float32), X_params.to(torch.float32), y[:,0]
            optimizer.zero_grad()
            out = model(X_maps, X_params)
            loss = criterion(out, y)
            losses.append(loss.item())
            pred = torch.argmax(out.detach(), dim=1)
            n_ac.append(torch.max(torch.unique(pred, return_counts=True)[1]).to(torch.float32))
            accuracy = (1*(pred==y)).to(torch.float32).mean()
            accuracies.append(accuracy)
            loss.backward()
            optimizer.step()
        acc_per_ac, test_n_ac, test_accuracies, test_t2accuracies, t3 = test(model, testloader)
        for ac in acc_per_ac.keys():
            writer.add_scalar(f"confusion/test_acc_{ac}", torch.tensor(acc_per_ac[ac]).mean(), epoch)
        writer.add_scalar("charts/test_max_freq",torch.mean(torch.tensor(test_n_ac)/testloader.batch_size), epoch)
        writer.add_scalar("charts/test_acc", torch.mean(torch.tensor(test_accuracies)), epoch)
        writer.add_scalar("charts/test_top_2_acc", torch.mean(torch.tensor(test_t2accuracies)), epoch)
        writer.add_scalar("charts/training_loss", torch.mean(torch.tensor(losses)), epoch)
        writer.add_scalar("charts/training_acc", torch.mean(torch.tensor(accuracies)), epoch)
        writer.add_scalar("charts/trainig_max_freq", torch.mean(torch.tensor(n_ac)/batch_size), epoch)
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], epoch)
        scheduler.step()
    return model

In [9]:
batch_size=512
model = train(traindata, testdata, batch_size=batch_size)

C:\Users\lenna\AppData\Local\Temp\ipykernel_8004\609556863.py:3: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(m.weight)
c:\Users\lenna\anaconda3\envs\lux\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
100%|██████████████████████████████████████████████████████████████████████████████| 100/100 [2:13:54<00:00, 80.35s/it]


In [11]:
torch.save(model.state_dict(), "models/aimbot4")

In [12]:
model = ActionOracle(7, 6)
check = torch.load("models/aimbot4")
model.load_state_dict(check)

C:\Users\lenna\AppData\Local\Temp\ipykernel_8004\609556863.py:3: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(m.weight)


<All keys matched successfully>

In [13]:
testloader = torch.utils.data.DataLoader(testdata, batch_size=512, shuffle=False)
acc_per_ac, test_n_ac, test_accuracies, test_t2accuracies, test_t3accuracies = test(model, testloader) 

In [14]:
print(torch.tensor(test_accuracies).mean())
print(torch.tensor(test_t2accuracies).mean())
print(torch.tensor(test_t3accuracies).mean())

tensor(0.5930)
tensor(0.7552)
tensor(0.8837)


In [15]:

for ac in acc_per_ac.keys():
    print(torch.tensor(acc_per_ac[ac]).mean())

tensor(0.7072)
tensor(0.3270)
tensor(0.4588)
tensor(0.4404)
tensor(0.2881)


In [14]:
a = 100
for i in range(5):
    a = a-0.12*a
    print(a)

88.0
77.44
68.1472
59.969536
52.77319168
